In [1]:
import os
import sys
from pathlib import Path

import yaml
from pyspark.sql import functions as F

# Find project root automatically
project_root = Path.cwd()
while not (project_root / "configs").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
os.chdir(project_root)

Project root: /home/ubuntu/renewable-energy-forecasting-pipeline


In [2]:
PROJECT_USER_CONFIG = os.environ.get("PROJECT_USER_CONFIG")

with open(project_root / PROJECT_USER_CONFIG, "r") as f:
    user_config = yaml.safe_load(f)

bucket = user_config["aws"]["project_bucket"]
gold_prefix = user_config["aws"]["gold_prefix"]

gold_root = f"s3a://{bucket}/{gold_prefix}"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("layer7_local_analysis")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.InstanceProfileCredentialsProvider"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d39cb6a1-a711-48b2-aaee-8631455b2a14;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 468ms :: artifacts dl 19ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------

In [3]:
region_monthly_path = f"{gold_root}/region/monthly"

region_monthly = spark.read.parquet(region_monthly_path)

print("Loaded region_monthly")
region_monthly.printSchema()

26/05/04 01:00:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Loaded region_monthly
root
 |-- month: integer (nullable = true)
 |-- monthly_region_capacity_factor: double (nullable = true)
 |-- monthly_mean_wind_speed_ms: double (nullable = true)
 |-- min_daily_region_capacity_factor: double (nullable = true)
 |-- max_daily_region_capacity_factor: double (nullable = true)
 |-- daily_observation_count: long (nullable = true)
 |-- avg_station_count: double (nullable = true)
 |-- is_valid_monthly_region_index: boolean (nullable = true)
 |-- wind_power_class: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- state: string (nullable = true)



In [4]:
paths = {
    "monthly_state": f"{gold_root}/analytics/monthly_state",
    "daily_region": f"{gold_root}/analytics/daily_region",
    "extreme_events": f"{gold_root}/analytics/extreme_events",
    "ml_base": f"{gold_root}/ml/base",
}

paths

{'monthly_state': 's3a://syed-datsbd-s2026/gold/wind/analytics/monthly_state',
 'daily_region': 's3a://syed-datsbd-s2026/gold/wind/analytics/daily_region',
 'extreme_events': 's3a://syed-datsbd-s2026/gold/wind/analytics/extreme_events',
 'ml_base': 's3a://syed-datsbd-s2026/gold/wind/ml/base'}

In [5]:
monthly_state = spark.read.parquet(paths["monthly_state"])
daily_region = spark.read.parquet(paths["daily_region"])
extreme_events = spark.read.parquet(paths["extreme_events"])
ml_base = spark.read.parquet(paths["ml_base"])

print("All final tables loaded.")

All final tables loaded.


In [6]:
tables = {
    "monthly_state": (monthly_state, ["state", "year", "month"]),
    "daily_region": (daily_region, ["state", "date_utc"]),
    "extreme_events": (extreme_events, ["state", "date_utc"]),
    "ml_base": (ml_base, ["state", "date_utc"]),
}

for name, (df, grain_cols) in tables.items():
    print(f"\n{name}")
    df.select(
        F.count("*").alias("rows"),
        F.countDistinct(*grain_cols).alias("distinct_grain"),
        F.countDistinct("state").alias("states"),
        F.countDistinct("year").alias("years"),
    ).show()


monthly_state


+-----+--------------+------+-----+
| rows|distinct_grain|states|years|
+-----+--------------+------+-----+
|17664|         17664|    48|   31|
+-----+--------------+------+-----+


daily_region


+------+--------------+------+-----+
|  rows|distinct_grain|states|years|
+------+--------------+------+-----+
|537449|        537449|    48|   31|
+------+--------------+------+-----+


extreme_events


+------+--------------+------+-----+
|  rows|distinct_grain|states|years|
+------+--------------+------+-----+
|537449|        537449|    48|   31|
+------+--------------+------+-----+


ml_base


+------+--------------+------+-----+
|  rows|distinct_grain|states|years|
+------+--------------+------+-----+
|537401|        537401|    48|   31|
+------+--------------+------+-----+



In [7]:
monthly_state.select(
    F.min("monthly_region_capacity_factor").alias("min_monthly_cf"),
    F.max("monthly_region_capacity_factor").alias("max_monthly_cf"),
).show()

daily_region.select(
    F.min("daily_region_capacity_factor").alias("min_daily_cf"),
    F.max("daily_region_capacity_factor").alias("max_daily_cf"),
).show()

ml_base.select(
    F.min("next_day_daily_region_capacity_factor").alias("min_target"),
    F.max("next_day_daily_region_capacity_factor").alias("max_target"),
    F.sum(F.col("next_day_daily_region_capacity_factor").isNull().cast("int")).alias("null_target"),
).show()

+--------------+--------------+
|min_monthly_cf|max_monthly_cf|
+--------------+--------------+
|      0.001662|      0.274203|
+--------------+--------------+



+------------+------------+
|min_daily_cf|max_daily_cf|
+------------+------------+
|         0.0|    0.900287|
+------------+------------+



+----------+----------+-----------+
|min_target|max_target|null_target|
+----------+----------+-----------+
|       0.0|  0.900287|          0|
+----------+----------+-----------+



In [8]:
extreme_events.groupBy("extreme_event_type").count().show()

+------------------+------+
|extreme_event_type| count|
+------------------+------+
|         high_wind| 53795|
|            normal|429917|
|          low_wind| 53737|
+------------------+------+



In [9]:
ml_base.select(
    F.sum(F.col("cf_lag_1d").isNull().cast("int")).alias("null_lag_1d"),
    F.sum(F.col("cf_lag_7d").isNull().cast("int")).alias("null_lag_7d"),
).show()

+-----------+-----------+
|null_lag_1d|null_lag_7d|
+-----------+-----------+
|         48|        336|
+-----------+-----------+



In [10]:
spark.stop()

## Final Validation Summary

In this notebook, we performed **post-build validation** of all final Layer 7 Gold tables stored in S3:

- `gold_monthly_state_wind`
- `gold_daily_region_wind`
- `gold_extreme_event_windows`
- `gold_ml_base_wind`

### Key checks performed

- **Row counts and grain validation**
  - All tables have **correct and unique grains**
  - No duplicate records at their intended level (state-month or state-date)

- **Coverage validation**
  - 48 U.S. states
  - 31 years of data
  - Consistent across all tables

- **Range checks**
  - Capacity factor values are within expected physical bounds:
    - Daily: 0.0 → 0.90
    - Monthly: 0.0016 → 0.27

- **Extreme event validation**
  - Distribution is balanced and realistic:
    - ~10% high wind
    - ~10% low wind
    - ~80% normal

- **ML table validation**
  - No missing target values
  - Expected lag nulls:
    - 1-day lag → 48 rows (one per state)
    - 7-day lag → 336 rows

### Conclusion

All Gold tables are:

- **complete**
- **consistent**
- **physically valid**
- **ready for analysis, visualization, and modeling**

This confirms that the pipeline successfully produces **reliable, production-quality datasets**.